# 02 — Smoke-train the model

Single-GPU training on a small subset for a handful of epochs. This is **not** how the paper's
models were trained — those used 12× V100s for 300 epochs (see `internal/slurm/train/`). The goal
of this notebook is to verify the full training loop runs on your machine and to give you a
starting template.

Three loss configurations match the paper:

| Run | `--lambda-residual` | `--lambda-phase` |
|---|---|---|
| FM only | 0 | 0 |
| FM + phase | 0 | 0.1 |
| FM + phase + residual | 1.0 | 0.1 |

We pick the third (the strongest) for the demo.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
MODEL_DIR = REPO_ROOT / 'Model'
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

DATA_ROOT = REPO_ROOT / 'Data' / 'demo_sweep'  # output of 01_dataset_generation.ipynb
CKPT_DIR  = REPO_ROOT / 'checkpoints' / 'demo'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('data:', DATA_ROOT)
print('ckpts will go to:', CKPT_DIR)

## Launch via CLI

`Model/train.py` is the canonical entry point. The cell below builds the command for a 5-epoch
smoke run on whatever data is in `DATA_ROOT`.

In [ ]:
cmd = [
    'python', str(MODEL_DIR / 'train.py'),
    '--data-root', str(DATA_ROOT),
    '--epochs', '5',
    '--batch-size', '2',
    '--lr', '1e-4',
    '--phaseA-epochs', '1',
    '--phaseB-epochs', '2',
    '--lambda-residual', '1.0',
    '--residual-warmup-epochs', '1',
    '--lambda-phase', '0.1',
    '--phase-warmup-epochs', '1',
    '--ckpt-every', '5',
    '--eval-every', '5',
    '--no-use-wandb',
]
print(' \\\n  '.join(cmd))

In [ ]:
import subprocess
# Uncomment to launch. Expect ~10 min on a single A100 with the demo dataset.
# subprocess.run(cmd, check=True)

## Loss curves

`train.py` writes a `history.json` per run. Read and plot it once training finishes.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = CKPT_DIR.parent / 'physics_unet_pbfm_ddp' / 'history.json'
if not history_path.is_file():
    print('No history.json yet — run the training cell first.')
else:
    h = json.loads(history_path.read_text())
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for key in ('train/fm', 'train/residual', 'train/phase'):
        if key in h:
            ax.plot(h[key], label=key)
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.set_yscale('log'); ax.legend()
    plt.tight_layout(); plt.show()

After a checkpoint is written, jump to [`03_inference.ipynb`](03_inference.ipynb) to evaluate it.